## 1. Importación y configuración

Se importan las funciones de preprocesamiento, los modelos y los parámetros del proyecto. También se vuelve a cargar el módulo del informe para utilizar su versión actualizada.

In [1]:
import sys
from pathlib import Path

working_dir = Path.cwd().resolve()
src_candidates = (working_dir / "src", working_dir.parent / "src")
src_path = next((path for path in src_candidates if path.is_dir()), None)
if src_path is None:
    raise FileNotFoundError("No se encontró la carpeta src en el proyecto ni en su directorio padre")
sys.path.insert(0, str(src_path))

from config import ARTIFACT_THRESHOLD, CHANNELS_TO_DROP, CONFIDENCE_THRESHOLD
from utils import process_patient
from model_utils import *
import importlib
import report
importlib.reload(report)
generate_report = report.generate_report


c:\Users\Victor\miniconda3\envs\ads\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Carga y preprocesamiento de los registros

Se recorren los diez participantes de la Cohorte III. Para cada uno, se carga el registro y las anotaciones de ambos expertos, se preprocesa la señal y se extraen sus características. El resultado se guarda junto con las etiquetas en `data_list`, que se usará en las siguientes etapas. Se asume que los datos están organizados en la carpeta `data/` según las instrucciones del README.

In [2]:

data_list = []
for patient_id in range(1, 11):
    print(f"Processing patient {patient_id}...")
    features, labels_1, labels_2 = process_patient(
        patient_id,
        CHANNELS_TO_DROP,
        artifact_threshold=ARTIFACT_THRESHOLD,
        show_plot=False,
        base_path=None,
    )
    data_list.append({'id': patient_id, 'X': features, 'y1': labels_1, 'y2': labels_2})

print(f"\nProcessing complete. Patients loaded: {len(data_list)}")

Processing patient 1...
Artifact-contaminated epochs: 2 / 954
Amplitude threshold: 8.815065480714984e-05
Processing patient 2...
Artifact-contaminated epochs: 3 / 941
Amplitude threshold: 9.35445788457238e-05
Processing patient 3...
Artifact-contaminated epochs: 1 / 824
Amplitude threshold: 9.726846513790948e-05
Processing patient 4...
Artifact-contaminated epochs: 8 / 794
Amplitude threshold: 8.3761253821896e-05
Processing patient 5...
Artifact-contaminated epochs: 24 / 944
Amplitude threshold: 8.959555661713269e-05
Processing patient 6...
Artifact-contaminated epochs: 1 / 853
Amplitude threshold: 8.028694339139756e-05
Processing patient 7...
Artifact-contaminated epochs: 5 / 814
Amplitude threshold: 0.0001131204593973564
Processing patient 8...
Artifact-contaminated epochs: 6 / 1000
Amplitude threshold: 8.763438205512885e-05
Processing patient 9...
Artifact-contaminated epochs: 7 / 969
Amplitude threshold: 0.00010306786452285627
Processing patient 10...
Artifact-contaminated epochs: 

## 3. Preparación de conjuntos y modelo multiclase

Se separan los participantes en entrenamiento (IDs 1 a 7) y prueba (IDs 8 a 10) de acuerdo al valor de `train_patinents`, conservando las épocas en las que coinciden ambos anotadores. Después se busca la mejor configuración del clasificador XGBoost de cuatro clases, se calibra y se obtienen predicciones y probabilidades para el conjunto de prueba.

In [3]:
train_patients = 7
X_train, X_test, y_train, y_test, y_train_original, y_test_original = data_preparation(data_list, train_patients=train_patients)  # Keep original labels for the binary model.
grid_search = xgb_definition()
calibrated_model = xgb_fitting(X_train, y_train, grid_search)
multiclass_predictions = calibrated_model.predict(X_test)
multiclass_probabilities = calibrated_model.predict_proba(X_test)

Fitting 3 folds for each of 4 candidates, totalling 12 fits
Best parameters: {'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 200, 'subsample': 0.8}


## 4. Modelo binario para distinguir N1 y N2

A partir de los mismos conjuntos, se seleccionan las épocas etiquetadas como N1 o N2 y se entrenan y calibran un segundo modelo XGBoost para distinguirlas. Sus probabilidades se convierten en etiquetas usando un umbral de 0,5.

In [4]:
X_train_bin, y_train_bin, X_test_bin, y_test_bin = binary_data_preparation(
    X_train, X_test, y_train_original, y_test_original
)
grid_search_bin = binary_xgb_definition()
calibrated_model_bin = binary_xgb_fitting(
    grid_search_bin, X_train_bin, y_train_bin
)
binary_probabilities = calibrated_model_bin.predict_proba(X_test_bin)
binary_threshold = 0.5
binary_predictions = (binary_probabilities[:, 1] > binary_threshold).astype(int)

Fitting 3 folds for each of 4 candidates, totalling 12 fits
Best parameters: {'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 100, 'subsample': 0.8}


## 5. Combinación de predicciones y confianza

Se combinan las probabilidades del modelo multiclase y del modelo binario para obtener predicciones de las cinco fases del sueño. Después se calcula una máscara que identifica las predicciones cuya probabilidad máxima supera `CONFIDENCE_THRESHOLD`; se muestra cuántas predicciones cumplen ese criterio.

In [5]:
combined_predictions, combined_probabilities, _ = model_combination(
    X_test, calibrated_model, calibrated_model_bin
)
confidence_mask = get_confidence_mask(
    combined_probabilities, threshold=CONFIDENCE_THRESHOLD
)
print(f"Predictions retained by confidence threshold: {confidence_mask.sum()} / {len(confidence_mask)}")

Predictions retained by confidence threshold: 1336 / 2657


## 6. Generación del informe de evaluación

Se reúnen las etiquetas reales, las predicciones y las probabilidades de ambos modelos y del modelo combinado. Con estos resultados, `generate_report` crea el informe HTML con las métricas, las curvas ROC y las matrices de confusión, incluyendo la evaluación filtrada por confianza.

In [ ]:
generate_report(
    y_true_multiclass=y_test,
    y_pred_multiclass=multiclass_predictions,
    probabilities_multiclass=multiclass_probabilities,
    y_true_binary=y_test_bin,
    y_pred_binary=binary_predictions,
    probabilities_binary=binary_probabilities[:, 1],
    y_true_combined=y_test_original,
    y_pred_combined=combined_predictions,
    probabilities_combined=combined_probabilities,
    confidence_mask=confidence_mask,
    confidence_threshold=CONFIDENCE_THRESHOLD,
    artifact_threshold=ARTIFACT_THRESHOLD,
    train_patient_ids=[p["id"] for p in data_list if p["id"] <= train_patients],
    test_patient_ids=[p["id"] for p in data_list if p["id"] > train_patients],
)